In [1]:
import sys
import os
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

from einops.layers.torch import Rearrange
from einops import rearrange, repeat
from fvcore.nn import FlopCountAnalysis, flop_count_table


In [2]:
sys.path.append(os.path.abspath(os.path.join('..', 'modules')))
from Trainer import Trainer

In [ ]:
def sample_time_range(self) -> Tuple[pd.Timestamp, pd.Timestamp, int]:
    """
    Sample random start and end times according to the provided ranges.
    """
    num_predict_hours = np.random.randint(*self.predict_sample_range)

        # see visualisation at Zeus/static/era5_start_offset_distribution.png
    if np.random.rand() > self.uniform_start_prob:
        start_offset = min(
            self.start_offset_range[1], # don't overshoot
            np.abs(
                int(np.random.normal(0, self.start_sample_std))
            ) + self.start_offset_range[0]
        )
    else:
        start_offset = int(np.random.uniform(*self.start_offset_range))

    start_timestamp = get_today("h") + pd.Timedelta(hours=start_offset)
    end_timestamp = start_timestamp + pd.Timedelta(hours=num_predict_hours - 1)

    return start_timestamp, end_timestamp, num_predict_hours

In [ ]:
def get_curved_scores(raw_scores: List[float], gamma: float) -> List[float]:
    """
    Given a list of raw float scores (can by any range),
    normalise them to 0-1 scores,
    and apply gamma correction to curve accordingly.

    This function assumes lower is better
    """
    min_score = min(raw_scores)
    max_score = max(raw_scores)

    result = []
    for score in raw_scores:
        if max_score == min_score:
            result.append(1.0) # edge case, avoid division by 0
        else:
            norm_rmse = (max_score - score) / (max_score - min_score)
            result.append(np.power(norm_rmse, gamma)) # apply gamma correction
    
    return result

In [ ]:
def set_rewards(
    output_data: torch.Tensor,
    miners_data: List[MinerData],
    baseline_data: Optional[torch.Tensor],
    difficulty_grid: np.ndarray,
    min_sota_delta: float
) -> List[MinerData]:
    """
    Calculates rewards for miner predictions based on RMSE and relative difficulty.
    NOTE: it is assumed penalties have already been scored and filtered out, 
      if not will do so without scoring those

    Args:
        output_data (torch.Tensor): The ground truth data.
        miners_data (List[MinerData]): List of MinerData objects containing predictions.
        baseline_data (torch.Tensor): OpenMeteo prediction, where additional incentive is awarded to beat this!
        difficulty_grid (np.ndarray): Difficulty grid for each coordinate.

    Returns:
        List[MinerData]: List of MinerData objects with updated rewards and metrics.
    """
    miners_data = [m for m in miners_data if not m.shape_penalty]

    if len(miners_data) == 0:
        return miners_data

    # old challenges have no baseline, use 0 to make it not affect scoring.
    baseline_rmse = 0
    if baseline_data is not None:
        baseline_rmse = rmse(output_data, baseline_data)
        
    avg_difficulty = difficulty_grid.mean()
    # make difficulty [-1, 1], then go between [1/scaler, scaler]
    gamma = np.power(REWARD_DIFFICULTY_SCALER, avg_difficulty * 2 - 1)

    # compute unnormalised scores
    for miner_data in miners_data:
        miner_data.rmse = rmse(output_data, miner_data.prediction)
        improvement = baseline_rmse - miner_data.rmse - min_sota_delta
        miner_data.baseline_improvement = max(0, improvement)

    quality_scores = get_curved_scores([m.rmse for m in miners_data], gamma)
    # negative since curving assumes minimal is the best
    improvement_scores = get_curved_scores([-m.baseline_improvement for m in miners_data], gamma)

    for miner_data, quality, improvement in zip(miners_data, quality_scores, improvement_scores):
        miner_data.reward = (1 - REWARD_IMPROVEMENT_WEIGHT) * quality + REWARD_IMPROVEMENT_WEIGHT * improvement

    return miners_data

In [3]:
config = {
    # image h w c
    'height': 72,
    'width': 144,
    'channels': 2,
    # video length in and out
    'T1': 1,    # 5 later.
    'MaxT2': 5,
    'FG': 24,   # frames in a frame group
    # patch size
    'patch_size': 8,
    # transformer
    'heads': 8,
    'dim_head': 32,
    'dropout': 0.0,
    'attn_dropout': 0.0,
    'drop_path': 0.0,
    'scale_dim': 4,
    # layers
    'attLayers' : 1,
    'zeusLayers': 6,
}

In [4]:
trainer = Trainer(config)

In [5]:
x = torch.rand(1, 24, 2, 72, 144)
output = trainer.model(x)
print(output.shape)  # [B, T, C, H, W]
# # Calculate FLOPs
flops = FlopCountAnalysis(trainer.model, x)
print(f'Number of flops: {flop_count_table(flops)}')


torch.Size([1, 24, 2, 72, 144])
Number of flops: | module                              | #parameters or shape   | #flops    |
|:------------------------------------|:-----------------------|:----------|
| model                               | 7.947M                 | 35.341G   |
|  to_patch_embedding.1               |  16.512K               |  63.701M  |
|   to_patch_embedding.1.weight       |   (128, 128)           |           |
|   to_patch_embedding.1.bias         |   (128,)               |           |
|  blocks                             |  7.913M                |  35.199G  |
|   blocks.0                          |   1.319M               |   5.866G  |
|    blocks.0.ts_temporal_transformer |    0.33M               |    1.329G |
|    blocks.0.ts_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_temporal_transformer |    0.33M               |    1.329G |
|   blocks.1               